### I Load environment variables

In [1]:
from dotenv import load_dotenv
import os 

load_dotenv()

SERPAPI_KEY = os.getenv("SERPAPI_KEY")
print("Key loaded:", SERPAPI_KEY[:8] + "...")

Key loaded: 39e0a96f...


### Define your product basket

In [2]:
products = [
    {"name": "Dell XPS 13",           "query": "Dell XPS 13 laptop",          "category": "laptop"},
    {"name": "MacBook Air M3",         "query": "Apple MacBook Air M3",         "category": "laptop"},
    {"name": "HP Spectre x360",        "query": "HP Spectre x360 laptop",       "category": "laptop"},
    {"name": "iPhone 15",              "query": "Apple iPhone 15 smartphone",   "category": "phone"},
    {"name": "Samsung Galaxy S24",     "query": "Samsung Galaxy S24 phone",     "category": "phone"},
    {"name": "Samsung Galaxy A54",     "query": "Samsung Galaxy A54 phone",     "category": "phone"},
    {"name": "GoPro Hero 13",          "query": "GoPro Hero 13 action camera",  "category": "camera"},
    {"name": "DJI Osmo Action",        "query": "DJI Osmo Action camera",       "category": "camera"},
]

### Single product test - test with 1 product first 

In [3]:
import serpapi

# Initialize client once (outside function)
client = serpapi.Client(api_key=os.getenv("SERPAPI_KEY"))

def fetch_product(product_name: str, country_code: str = "nz", num_results: int = 3):
    results = client.search({
        "engine": "google_shopping",
        "q": product_name,
        "gl": country_code,
        "hl": "en"
    })
    
    return results.get("shopping_results", [])[:num_results]

In [4]:
# Test with 1 product 
raw = fetch_product("Dell XPS 13")
print(f"Fetched {len(raw)} results")

Fetched 3 results


In [5]:
import json

print(json.dumps(raw[0], indent=2))

{
  "position": 1,
  "title": "Dell XPS 3K OLED cxn9345cto04mnzh",
  "product_id": "17439425737950475199",
  "product_link": "https://www.google.com/search?ibp=oshop&q=Dell XPS 13&prds=catalogid:17439425737950475199,headlineOfferDocid:4178638582499381924,imageDocid:12514492040275843381,rds:PC_12164043393717737707|PROD_PC_12164043393717737707,gpcid:12164043393717737707,mid:576462829857132261,pvt:hg&hl=en&gl=nz&udm=28",
  "immersive_product_page_token": "eyJlaSI6IllpSGdhYkdKTjRqTGtQSVB6OXpueVFJIiwicHJvZHVjdGlkIjoiIiwiY2F0YWxvZ2lkIjoiMTc0Mzk0MjU3Mzc5NTA0NzUxOTkiLCJoZWFkbGluZU9mZmVyRG9jaWQiOiI0MTc4NjM4NTgyNDk5MzgxOTI0IiwiaW1hZ2VEb2NpZCI6IjEyNTE0NDkyMDQwMjc1ODQzMzgxIiwicmRzIjoiUENfMTIxNjQwNDMzOTM3MTc3Mzc3MDd8UFJPRF9QQ18xMjE2NDA0MzM5MzcxNzczNzcwNyIsInF1ZXJ5IjoiRGVsbCtYUFMrMTMiLCJncGNpZCI6IjEyMTY0MDQzMzkzNzE3NzM3NzA3IiwibWlkIjoiNTc2NDYyODI5ODU3MTMyMjYxIiwicHZ0IjoiaGciLCJ1dWxlIjpudWxsLCJnbCI6Im56IiwiaGwiOiJlbiJ9",
  "serpapi_immersive_product_api": "https://serpapi.com/search.json?engine=googl

### Extract only important fields

In [24]:
from datetime import datetime, timezone 
import pytz

def parse_product(result: dict, category: str, product_name: str) -> dict:
    nzt = pytz.timezone("Pacific/Auckland")
    return {
        "product_name": product_name,
        "category":     category,
        "title":        result.get("title"),
        "price":        result.get("extracted_price"),
        "old_price":    float(result["extracted_old_price"]) if result.get("extracted_old_price") else None,
        "seller":       result.get("source"),
        "rating":       result.get("rating"),
        "reviews":      result.get("reviews"),
        "position":     result.get("position"),
        "ingested_at":  datetime.now(nzt).isoformat(),
    }

In [7]:
# Test with the raw result
parsed = [parse_product(r, "laptop", "Dell XPS 13") for r in raw]

for p in parsed:
    print(p)

{'product_name': 'Dell XPS 13', 'category': 'laptop', 'title': 'Dell XPS 3K OLED cxn9345cto04mnzh', 'price': 2899.15, 'old_price': None, 'seller': 'Dell New Zealand', 'rating': 3.9, 'reviews': 1100, 'position': 1, 'ingested_at': '2026-04-16T12:18:25.839038+12:00'}
{'product_name': 'Dell XPS 13', 'category': 'laptop', 'title': 'XPS 9320 Plus Laptop', 'price': 2597.85, 'old_price': None, 'seller': 'dell.com', 'rating': 3.8, 'reviews': 2100, 'position': 2, 'ingested_at': '2026-04-16T12:18:25.839067+12:00'}
{'product_name': 'Dell XPS 13', 'category': 'laptop', 'title': 'Dell Xps 13-9350 -8Gb Ram, 256 Gb Storage', 'price': 149.0, 'old_price': None, 'seller': 'Trade Me', 'rating': None, 'reviews': None, 'position': 3, 'ingested_at': '2026-04-16T12:18:25.839076+12:00'}


### Fetch all 8 products (full basket test)

In [8]:
all_results = []

for product in products:
    print(f"Fetching: {product['name']}...")
    raw_results = fetch_product(product["query"], num_results=3)  
    parsed_results = [
        parse_product(r, product["category"], product["name"])
        for r in raw_results
    ]
    all_results.extend(parsed_results)

print(f"\nTotal records fetched: {len(all_results)}")

Fetching: Dell XPS 13...
Fetching: MacBook Air M3...
Fetching: HP Spectre x360...
Fetching: iPhone 15...
Fetching: Samsung Galaxy S24...
Fetching: Samsung Galaxy A54...
Fetching: GoPro Hero 13...
Fetching: DJI Osmo Action...

Total records fetched: 24


### Preview as DataFrame

In [9]:
import pandas as pd

df = pd.DataFrame(all_results)
df.head(10)

,product_name,category,title,price,old_price,seller,rating,reviews,position,ingested_at
0,Dell XPS 13,laptop,Dell XPS 3K OLED cxn9345cto04mnzh,2899.15,None,Dell New Zealand,3.9,1100.0,1,2026-04-16T12:18:27.013372+12:00
1,Dell XPS 13,laptop,"Dell XPS 13 9350 Laptop, 13.4"" 3K OLED 60Hz To...",4260.06,None,Microless.com,4.1,675.0,2,2026-04-16T12:18:27.013428+12:00
2,Dell XPS 13,laptop,Dell XPS 13 9310 Touchscreen Laptop 13.4 inch ...,3664.05,None,Microless.com,4.0,8200.0,3,2026-04-16T12:18:27.013441+12:00
3,MacBook Air M3,laptop,"Apple Macbook Air 13"" Laptop with M3 Chip - Mi...",2399.00,None,PB Tech,4.7,3000.0,1,2026-04-16T12:18:27.092675+12:00
4,MacBook Air M3,laptop,"Apple MacBook Air 13"" Laptop with M3 Chip - Si...",2499.00,None,PB Tech,4.7,3000.0,2,2026-04-16T12:18:27.092796+12:00
5,MacBook Air M3,laptop,"Apple MacBook Air 13.6"" (2024 M3) 512GB (16GB ...",2159.82,None,MightyApe.co.nz,4.7,3000.0,3,2026-04-16T12:18:27.092830+12:00
6,HP Spectre x360,laptop,"HP Spectre x360 16-aa0001TU 16"" 2.8K OLED Touc...",3575.58,None,PB Tech,3.9,9.0,1,2026-04-16T12:18:27.171433+12:00
7,HP Spectre x360,laptop,HP Spectre x360 16-aa0009TX NVIDIA RTX 4050 Fl...,3829.50,None,PB Tech,3.3,6.0,2,2026-04-16T12:18:27.171527+12:00
8,HP Spectre x360,laptop,"Spectre x360 16-aa0001TU 16"" 2.8K OLED Touch F...",3295.75,None,Auckland Airport,4.0,10.0,3,2026-04-16T12:18:27.171544+12:00
9,iPhone 15,phone,Apple iPhone 15,1299.00,None,Noel Leeming,4.7,55000.0,1,2026-04-16T12:18:27.269680+12:00


### Basic Quality Check

In [10]:
print("=== Null counts ===")
print(df.isnull().sum())

print("\n=== Price range per product ===")
print(df.groupby("product_name")["price"].agg(["min", "max", "mean"]).round(2))

print("\n=== Unique sellers ===")
print(df["seller"].value_counts())

=== Null counts ===
product_name     0
category         0
title            0
price            0
old_price       24
seller           0
rating           1
reviews          1
position         0
ingested_at      0
dtype: int64

=== Price range per product ===
                        min      max     mean
product_name                                 
DJI Osmo Action      349.00   829.00   602.33
Dell XPS 13         2899.15  4260.06  3607.75
GoPro Hero 13        747.50   848.58   795.34
HP Spectre x360     3295.75  3829.50  3566.94
MacBook Air M3      2159.82  2499.00  2352.61
Samsung Galaxy A54   419.99   509.00   460.39
Samsung Galaxy S24   804.99  2199.00  1283.82
iPhone 15            888.99  1299.00  1032.33

=== Unique sellers ===
seller
PB Tech                        7
Trade Me                       3
Microless.com                  2
MightyApe.co.nz                2
Dick Smith NZ                  2
Dell New Zealand               1
Auckland Airport               1
Noel Leeming          

In [11]:
# Check what the raw response actually contains
raw_check = fetch_product("Dell XPS 13 laptop", num_results=3)

for i, r in enumerate(raw_check):
    print(f"--- Result {i+1} ---")
    print(f"title          : {r.get('title')}")
    print(f"price          : {r.get('price')}")
    print(f"extracted_price: {r.get('extracted_price')}")
    print(f"old_price      : {r.get('old_price')}")
    print(f"extracted_old_price: {r.get('extracted_old_price')}")
    print()

--- Result 1 ---
title          : Dell XPS 3K OLED cxn9345cto04mnzh
price          : $2,899.15
extracted_price: 2899.15
old_price      : None
extracted_old_price: None

--- Result 2 ---
title          : Dell XPS 13 9350 Laptop, 13.4" 3K OLED 60Hz Touch Display, Intel Core Ultra 9 288V, 32GB RAM, 2TB SSD, Intel Arc Graphics, English Keyboard, Windows 1
price          : $4,260.06
extracted_price: 4260.06
old_price      : None
extracted_old_price: None

--- Result 3 ---
title          : Dell XPS 13 9310 Touchscreen Laptop 13.4 inch Thin and Light. Intel Core
price          : $3,664.05
extracted_price: 3664.05
old_price      : None
extracted_old_price: None



#### This mean some of the products is not on the discount so we dont have old price, the pipeline is correct and not have issue.

## Test with dlt library

In [18]:
import dlt
@dlt.resource(name="shopping_results")

def shopping_resource():
    "Write a function to ingest data using dlt lib"

    for record in all_results: # all_results already implemented above
        yield record

pipeline = dlt.pipeline(
    pipeline_name = "electronic_retail_price",
    destination = "duckdb",
    dataset_name = "raw_shopping"
)

load_info = pipeline.run(shopping_resource())
print(load_info)

2026-04-16 12:20:50,441|[WARNING]|79753|8011931328|dlt|validate.py|verify_normalized_table:113|In schema `electronic_retail_price`: The following columns in table 'shopping_results' did not receive any data during this load and therefore could not have their types inferred:
  - old_price

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'old_price': {'data_type': 'text'}})



Pipeline electronic_retail_price load step completed in 0.05 seconds
1 load package(s) were loaded to destination duckdb and into dataset raw_shopping
The duckdb destination used duckdb:////Users/dazieldang/Desktop/Project/retail-data-engineer-pipeline/Notebooks /electronic_retail_price.duckdb location to store data
Load package 1776298850.3850272 is LOADED and contains no failed jobs


In [19]:
import duckdb

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

print("=== Tables created by dlt ===")
print(conn.execute("SHOW TABLES").fetchdf())

print("\n=== Sample data ===")
print(conn.execute("SELECT * FROM raw_shopping.shopping_results LIMIT 5").fetchdf())

print("\n=== Row count ===")
print(conn.execute("SELECT COUNT(*) FROM raw_shopping.shopping_results").fetchdf())

=== Tables created by dlt ===
Empty DataFrame
Columns: [name]
Index: []

=== Sample data ===
     product_name category                                              title  \
0     Dell XPS 13   laptop                  Dell XPS 3K OLED cxn9345cto04mnzh   
1     Dell XPS 13   laptop  Dell XPS 13 9350 Laptop, 13.4" 3K OLED 60Hz To...   
2     Dell XPS 13   laptop  Dell XPS 13 9310 Touchscreen Laptop 13.4 inch ...   
3  MacBook Air M3   laptop  Apple Macbook Air 13" Laptop with M3 Chip - Mi...   
4  MacBook Air M3   laptop  Apple MacBook Air 13" Laptop with M3 Chip - Si...   

     price            seller  rating  reviews  position  \
0  2899.15  Dell New Zealand     3.9     1100         1   
1  4260.06     Microless.com     4.1      675         2   
2  3664.05     Microless.com     4.0     8200         3   
3  2399.00           PB Tech     4.7     3000         1   
4  2499.00           PB Tech     4.7     3000         2   

                       ingested_at        _dlt_load_id         _d

### Check dlt schema inference

In [20]:
print("=== Schema inferred by dlt ===")
print(conn.execute("DESCRIBE raw_shopping.shopping_results").fetchdf())

=== Schema inferred by dlt ===
     column_name               column_type null   key default extra
0   product_name                   VARCHAR  YES  None    None  None
1       category                   VARCHAR  YES  None    None  None
2          title                   VARCHAR  YES  None    None  None
3          price                    DOUBLE  YES  None    None  None
4         seller                   VARCHAR  YES  None    None  None
5         rating                    DOUBLE  YES  None    None  None
6        reviews                    BIGINT  YES  None    None  None
7       position                    BIGINT  YES  None    None  None
8    ingested_at  TIMESTAMP WITH TIME ZONE  YES  None    None  None
9   _dlt_load_id                   VARCHAR   NO  None    None  None
10       _dlt_id                   VARCHAR   NO  None    None  None


This last cell is important — it shows you how dlt automatically inferred column types from your data. This is the schema that will eventually be applied when we switch destination to Postgres and S3.

In [23]:
### Save the dataframe as exploring dataset
explored_dataset = df

explored_dataset.to_csv("../data/explored-testing.csv")